In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import mode
from sklearn.model_selection import ParameterGrid, train_test_split

In [2]:
seed = 1234
np.random.seed(seed)

## Data Loading

In [7]:
DATA = pd.read_csv("data/proc_training_nobin.csv")
TEST_DATA = pd.read_csv("data/proc_testing_nobin.csv")

In [4]:
X = DATA.drop(columns='label')
y = DATA['label']

In [5]:
TEST_txkey = TEST_DATA['txkey']
X_test = TEST_DATA.drop(columns='txkey')

## Model Construction

In [6]:
# model = lgb.LGBMClassifier(
#     boosting_type='gbdt',
#     objective='binary',
#     device='gpu',
#     random_state=seed,
# )
# model.set_params(
#     bagging_freq=1,
#     bagging_fraction=0.9,
#     feature_fraction=0.9,
#     # feval=lgb_f1_score,
#     # evals_result=eval_res,
# )

## Training & Validation
先 grid search 找到不錯的參數，再去做 cross validation

In [8]:
cat_feats = [
    'contp',
    'etymd',
    'ecfg',
    # 'insfg',
    'bnsfg',
    'stscd',
    'ovrlt',
    'flbmk',
    'hcefg',
    'csmcu',
    'flg_3dsmk',
    # 'mcc_gp',
    # 'region_gp',
    # 'chid_gp'
]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3)

# model.fit(
#     X_train, y_train,
#     categorical_feature=cat_feats,
#     callbacks=[lgb.early_stopping(5)],
#     eval_set=[(X_valid, y_valid)],
#     eval_metric=['binary_logloss'],
# )

In [ ]:
params = {
    'num_leaves': range(45, 66, 2),
    'learning_rate': np.linspace(0.05, 0.15, 11),
    'n_estimators': 10,
    'min_split_gain': np.arange(0.00005, 0.00035, 0.00005),
    'min_child_samples': range(10, 40, 10),
}
grid = ParameterGrid(params)
best_params = None
best_score = 100

for params in grid:
    model = lgb.LGBMClassifier(
        boosting_type='gbdt',
        objective='binary',
        device='gpu',
        random_state=seed,
        **params
    )
    
    model.fit(
        X_train, y_train,
        categorical_feature=cat_feats,
        callbacks=[lgb.early_stopping(5)],
        eval_set=[(X_valid, y_valid)],
        eval_metric=['binary_logloss'],
    )
    
    score = model.best_score_['valid_0']['binary_logloss']
    if score < best_score:
        best_score = score
        best_params = params

[LightGBM] [Info] Number of positive: 22492, number of negative: 6059476
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1436
[LightGBM] [Info] Number of data points in the train set: 6081968, number of used features: 19
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 8 dense feature groups (46.40 MB) transferred to GPU in 0.040772 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003698 -> initscore=-5.596219
[LightGBM] [Info] Start training from score -5.596219
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[1]	valid_0's binary_logloss: 0.0149987
[LightGBM] [Info] Number of positive: 22492, number of negative: 6059476
[LightGBM] [Info] This is the GPU trainer!!

In [110]:
print(f'Best Score: {best_score}')
print(f'Best Params: {best_params}')

Best Score: 0.01451347663227057
Best Params: {'learning_rate': 0.1, 'min_child_samples': 20, 'min_split_gain': 0.0001, 'n_estimators': 50, 'num_leaves': 51}


In [111]:
# Found in the above grid search
# best_params = {
#     'learning_rate': 0.1,
#     'min_child_samples': 20,
#     'min_split_gain': 0.0001,
#     'n_estimators': 50,
#     'num_leaves': 51
# }

In [113]:
D_train = lgb.Dataset(X, y)

In [118]:
lgb_params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'device': 'gpu',
}
lgb_params.update(best_params)
eval_res = lgb.cv(
    lgb_params,
    D_train,
    nfold=5,
    metrics='binary_log_loss',
    categorical_feature=cat_feats,
    return_cvbooster=True
)
eval_res

/home/kuanhe/.cache/pypoetry/virtualenvs/esun-YNnw7uhi-py3.11/lib/python3.11/site-packages/lightgbm/engine.py:685: UserWarning: Found 'n_estimators' in params. Will use it instead of 'num_boost_round' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'num_boost_round' argument")


[LightGBM] [Info] Number of positive: 25623, number of negative: 6925197
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1444
[LightGBM] [Info] Number of data points in the train set: 6950820, number of used features: 19
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 8 dense feature groups (53.03 MB) transferred to GPU in 0.037067 secs. 1 sparse feature groups
[LightGBM] [Info] Number of positive: 25624, number of negative: 6925197
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1444
[LightGBM] [Info] Number of data points in the train set: 6950821, number of used features: 19
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[Ligh

{'cvbooster': <lightgbm.engine.CVBooster at 0x7f0f022d4650>}

In [129]:
models = eval_res['cvbooster']

## Predict

In [132]:
outputs = models.predict(X_test)
preds = [np.where(outputs[i] < 0.5, 0, 1) for i in range(len(outputs))]
y_pred = mode(preds)[0]

res = {
    'txkey': TEST_txkey,
    'label': y_pred
}

pd.DataFrame(res).to_csv('preds.csv', index=False)

In [133]:
pd.Series(y_pred).value_counts()

0    598654
1      1528
Name: count, dtype: int64